In [ ]:
!pip install crewai langchain groq gradio

In [ ]:
import os
from crewai import Agent, Task, Crew
from langchain_groq import ChatGroq
import gradio as gr

In [ ]:
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

In [ ]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.3
)

In [ ]:
monitor_agent = Agent(
    role="Network Monitoring Agent",
    goal="Continuously monitor avionics router bandwidth usage",
    backstory="Expert in real-time avionics telemetry systems.",
    llm=llm,
    verbose=True
)

In [ ]:
analysis_agent = Agent(
    role="Bandwidth Analysis Agent",
    goal="Analyze current network usage and detect congestion patterns",
    backstory="Specialist in network traffic analysis and anomaly detection.",
    llm=llm,
    verbose=True
)

In [ ]:
optimization_agent = Agent(
    role="Network Optimization Agent",
    goal="Optimize bandwidth allocation based on mission-critical priorities",
    backstory="Expert in avionics QoS, traffic shaping, and fail-safe routing.",
    llm=llm,
    verbose=True
)

In [ ]:
def create_task(input_data):
    return Task(
        description=f"""
        You are managing an avionics network router.

        Current Bandwidth Usage:
        {input_data}

        Goals:
        - Ensure flight-critical systems always get highest priority
        - Prevent congestion collapse
        - Optimize remaining bandwidth efficiently

        Decide:
        1. Which applications to prioritize
        2. Which to throttle
        3. Whether rerouting is required

        Provide a structured decision output.
        """,
        agent=optimization_agent,
        expected_output="Structured optimization decision with priorities, throttling, and routing strategy"
    )

In [ ]:
def run_crew(input_data):
    task = create_task(input_data)

    crew = Crew(
        agents=[monitor_agent, analysis_agent, optimization_agent],
        tasks=[task],
        verbose=True
    )

    result = crew.kickoff()
    return str(result)

In [ ]:
def optimize_bandwidth(input_text):
    return run_crew(input_text)

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("""
    # ✈️ Avionics Network Bandwidth Optimizer
    ### Goal-Based Agentic AI System (CrewAI + LLaMA 3.1)

    Enter current network usage data to get intelligent optimization decisions.
    """)

    with gr.Row():
        input_box = gr.Textbox(
            label="📡 Current Network Usage",
            placeholder="""
Example:
- Flight Control: 40%
- Navigation: 25%
- Passenger WiFi: 20%
- Maintenance Logs: 15%
            """,
            lines=8
        )

    output_box = gr.Textbox(
        label="⚙️ Optimization Decision",
        lines=12
    )

    optimize_btn = gr.Button("🚀 Optimize Bandwidth")

    optimize_btn.click(
        fn=optimize_bandwidth,
        inputs=input_box,
        outputs=output_box
    )

demo.launch()